# Implementation of A* and GBFS and comparison with Djikstra and Bellman-Ford
## CI2025_Lab3
**K/BIDI Thomas**

In [ ]:
from itertools import product, combinations
import numpy as np
import networkx as nx
import heapq
from itertools import product
import time
import pandas as pd
from IPython.display import display
from tqdm.notebook import tqdm

## Problems

We have a problem generator which create a graph in which we will try to find the shortest path between every nodes.  
The generator takes as arguments the following parameters : 
- **size** : The number of nodes in the graph (i.d. the number of cities).
- **density** : Describe of much are connected the cities to each other.
- **noise_level** : How much noise is added to the weights between the cities.
- **negative_values** : Can the graph contains negative values or not (which implies the potential presence of negative cycles).

Our objective will be to compare the implementation of Djikstra and Bellman-Ford from NX to our implementations of A* and GBFS, using the euclidean distance as their heuristic, for the following combinations of parameters :

```python
    sizes = [10, 20, 50, 100, 200, 500, 1000]
    densities = [0.2, 0.5, 0.8, 1.0]
    noise_levels = [0.0, 0.1, 0.5, 0.8]
    negative_values = [False, True]
```

In [ ]:
def create_problem(
    size: int,
    *,
    density: float = 1.0,
    negative_values: bool = False,
    noise_level: float = 0.0,
    seed: int = 42,
) -> np.ndarray:
    """Problem generator for Lab3"""
    rng = np.random.default_rng(seed)
    map = rng.random(size=(size, 2))
    problem = rng.random((size, size))
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    for a, b in product(range(size), repeat=2):
        if rng.random() < density:
            problem[a, b] += np.sqrt(
                np.square(map[a, 0] - map[b, 0]) + np.square(map[a, 1] - map[b, 1])
            )
        else:
            problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round(), map

## What is A* and GBFS ?

Firstly, let's present A* and GBFS.

**A\*** is a variant of Djikstra which use an heuristic to find the best way to the destination, it is supposed to be faster than Djikstra but it do not always return the optimal solution.  
**GBFS**, for **Greedy Breadth First Search**, is a variant of BFS using the heuristic as the greedy criterion, it do not care about the actual weights of the path and only use the heuristic to make its selection.

In [ ]:
# Return the euclidean distance between node1 and node2 on the map
def euclidean_distance(node1, node2, map):
    return np.linalg.norm(np.array(map[node1]) - np.array(map[node2]))

In [ ]:
# The implementation of A* is strongly inspired of the pseudo-code on this webpage https://en.wikipedia.org/wiki/A*_search_algorithm
def A_star(G, map, start, destination, heuristic):
    open_list = [(heuristic(start, destination, map), start)]
    heapq.heapify(open_list)
    open_set = {start} # For fast access

    came_from = {start: None}
    g_scores = {node : np.inf for node in G.nodes}
    f_scores = {node : np.inf for node in G.nodes}
    g_scores[start] = 0
    f_scores[start] = heuristic(start, destination, map)

    while open_list:
        _, current = heapq.heappop(open_list)

        if (current == destination):
            path = [current]
            while current in came_from:
                current = came_from[current]
                if current != None:
                    path.append(current)
            return path[::-1], g_scores[destination] # Reverse the path and return it with the its cost

        open_set.discard(current)

        for neighbor in G.neighbors(current):
            weight = G[current][neighbor]["weight"]

            if (weight < 0): # This addition avoid A* to be blocked in a negative cycle
                return None, -np.inf # Failure
            
            tentative_g_score = g_scores[current] + weight

            if (tentative_g_score < g_scores[neighbor]):
                came_from[neighbor] = current
                g_scores[neighbor] = tentative_g_score
                f_scores[neighbor] = tentative_g_score + heuristic(neighbor, destination, map)
                if (not(neighbor in open_set)):
                    heapq.heappush(open_list, (f_scores[neighbor], neighbor))
                    open_set.add(neighbor)
                    
    return None, np.inf # Failure

In [ ]:
# GBFS is a modification of the A* code which only take into account the heuristic
def GBFS(G, map, start, destination, heuristic):
    open_list = [(heuristic(start, destination, map), start)]
    heapq.heapify(open_list)
    open_set = {start} # For fast access

    came_from = {start: None}

    while open_list:
        _, current = heapq.heappop(open_list)

        if (current == destination):
            path = [current]
            while current in came_from:
                current = came_from[current]
                if current != None:
                    path.append(current)
            path = path[::-1] # Reverse the list 
            return path, nx.path_weight(G, path, weight='weight') # Return the path with the its cost

        open_set.discard(current)

        for neighbor in G.neighbors(current):
            if (not(neighbor in came_from)):
                came_from[neighbor] = current
                
                if (not(neighbor in open_set)):
                    heapq.heappush(open_list, (heuristic(neighbor, destination, map), neighbor))
                    open_set.add(neighbor)

    return None, np.inf # Failure

## Resolution loop

We will now create the main loop for our problem, it will execute Bellman-Ford, Djikstra, A* and GBFS on all the combination of the parameters listed above.  
Since their might be some negative cycles, when negative values are present, Djikstra error will be considered as (None, -inf), A* do the same thing and GBFS will try to return a path but its cost will not be valid since the negative cycles induce that there is not "shortest path".  

For this main loop to be generic, we will create some functions which will help us to easily add new algorithms.

**Disclaimer** : It takes a lot of time to run the program for every configurations !

In [ ]:
# This function will take another function as a parameter and run it with the additionnal parameters
def run(function, *args, **kwargs):
    t0 = time.perf_counter()
    error_occured = True
    try:
        path, cost = function(*args, **kwargs)
        error_occured = False
    except nx.NetworkXNoPath:
        # Nodes are not connected
        path = None
        cost = np.inf
    except nx.NetworkXUnbounded:
        # Negative cycle detected
        path = None
        cost = -np.inf
    except:
        # Probably an error from Djikstra (because it cannot detect the negative cycles)
        path = None
        cost = -np.inf
    
    t1 = time.perf_counter()
    error_occured = True if path == None else error_occured # No path found is considered as an error
    duration = -1 if error_occured else (t1 - t0) # Don't take into account the duration if there is an error

    return {"path" : path, "cost" : cost, "duration" : duration}

In [ ]:
# Here we create the functions to run Bellman-Ford and Djikstra using the run function defined above.
def bellman_ford(G, s, d):
    path = nx.bellman_ford_path(G, s, d, weight="weight")
    cost = nx.path_weight(G, path, "weight")
    return path, cost

def djikstra(G, s, d):
    path = nx.dijkstra_path(G, s, d, weight="weight")
    cost = nx.path_weight(G, path, "weight")
    return path, cost

In [ ]:
# Parameters of the configuration
sizes = [10, 20, 50, 100, 200, 500, 1000]
densities = [0.2, 0.5, 0.8, 1.0]
noise_levels = [0.0, 0.1, 0.5, 0.8]
negative_values = [False, True]
configurations = list(product(sizes, densities, noise_levels, negative_values))

# Here is the holders for the results
results = {}

for config in tqdm(configurations, desc="Number of configurations", unit="config"):
    size, density, noise_level, have_negative_values = config
    
    results[config] = {"Bellman_Ford" : [], "Djikstra" : [], "GBFS" : [], "A*" : []} # We create the entry for this configuration

    # We create the problem and the induced graph
    problem, map = create_problem(size, density=density, noise_level=noise_level, negative_values=have_negative_values)
    masked = np.ma.masked_array(problem, mask=np.isinf(problem))
    G = nx.from_numpy_array(masked, create_using=nx.DiGraph)

    for s, d in combinations(range(problem.shape[0]), 2):
        result = {"start": s, "dest": d}
        results[config]["Bellman_Ford"].append({**result, **run(bellman_ford, G, s, d)})
        results[config]["Djikstra"].append({**result, **run(djikstra, G, s, d)})
        results[config]["GBFS"].append({**result, **run(GBFS, G, map, s, d, euclidean_distance)})
        results[config]["A*"].append({**result, **run(A_star, G, map, s, d, euclidean_distance)})

## Visualization

Let's visualize for each configuration how each algorithm performed.  
To do so we are going to make a table with the mean cost, mean duration and success rate of each algorithm.  
(Keep in mind that GBFS will return a path even if there is no "shortest path", when negative cycles are present).

In [ ]:
rows = []

# Loading of the data as rows of the dataframe
for config, algos in results.items():
    size, density, noise_level, have_negative_values = config

    for algo, entries in algos.items():
        for entry in entries:
            cost = entry["cost"]
            duration = entry["duration"]
            success = int(not np.isinf(cost))
            
            rows.append({ "Algorithm" : algo,
                          "Size" : size,
                          "Density" : density,
                          "Noise" : noise_level,
                          "Negative_values" : have_negative_values,
                          "Cost" : cost if success else np.nan,
                          "Duration" : duration if success else np.nan,
                          "Success" : success })

df = pd.DataFrame(rows) # Dataframe's conversion

configurations = df.groupby(["Size", "Density", "Noise", "Negative_values"])

for config, subgroup in configurations:
    size, density, noise_level, have_negative_values = config
    print(f"Configuration -> Size : {size} | Density : {density} | Noise level : {noise_level} | Negative values : {have_negative_values}")

    summary = subgroup.groupby("Algorithm").agg({"Cost": "mean", "Duration": "mean", "Success": "mean"}).reset_index()
    summary.columns = ["Algorithm", "mean_cost", "mean_duration", "success_rate"]

    display(summary)